In [0]:
display(dbutils.fs.ls(
    "abfss://landing@batchstorage11.dfs.core.windows.net/"
))

In [0]:
sales_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("abfss://landing@batchstorage11.dfs.core.windows.net/sales.csv")
)

display(sales_df)

In [0]:
from pyspark.sql.functions import col, count, when

# Check total number of records
print("Total records:", sales_df.count())

# Check missing values in each column
sales_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in sales_df.columns
]).display()

# Check duplicate order IDs
print(
    "Duplicate order IDs:",
    sales_df.count() - sales_df.dropDuplicates(["order_id"]).count()
)

# Check invalid quantities
print(
    "Invalid quantities:",
    sales_df.filter(col("quantity") <= 0).count()
)

In [0]:
# Check invalid unit prices
invalid_prices = sales_df.filter(col("unit_price") <= 0).count()

print("Invalid unit prices:", invalid_prices)

In [0]:
bronze_path = (
    "abfss://bronze@batchstorage11.dfs.core.windows.net/sales"
)

sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(bronze_path)

print("Sales data saved to Bronze successfully!")

In [0]:
bronze_df = spark.read.format("delta").load(
    "abfss://bronze@batchstorage11.dfs.core.windows.net/sales"
)

display(bronze_df)

In [0]:
from pyspark.sql.functions import col, trim

bronze_path = "abfss://bronze@batchstorage11.dfs.core.windows.net/sales"
silver_path = "abfss://silver@batchstorage11.dfs.core.windows.net/sales"

# Read Bronze data
bronze_df = spark.read.format("delta").load(bronze_path)

# Clean and validate the data
silver_df = (
    bronze_df
    .dropDuplicates(["order_id"])
    .filter(
        col("order_id").isNotNull()
        & col("customer_id").isNotNull()
        & col("quantity").isNotNull()
        & col("unit_price").isNotNull()
    )
    .withColumn("product", trim(col("product")))
)

# Write cleaned data to Silver
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

print("Silver transformation completed!")
print("Silver record count:", silver_df.count())

In [0]:
from pyspark.sql.functions import sum, round

silver_path = "abfss://silver@batchstorage11.dfs.core.windows.net/sales"
gold_path = "abfss://gold@batchstorage11.dfs.core.windows.net/sales_summary"

# Read Silver data
silver_df = spark.read.format("delta").load(silver_path)

# Calculate sales summary by product
gold_df = (
    silver_df
    .withColumn("sales_amount", round(silver_df["quantity"] * silver_df["unit_price"], 2))
    .groupBy("product")
    .agg(
        sum("quantity").alias("total_quantity"),
        sum("sales_amount").alias("total_sales")
    )
)

# Write results to Gold
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path)

display(gold_df)

## Sales Data Pipeline – Execution Completed
The sales data pipeline has completed the following stages:

- **Landing**: Read sales data from `sales.csv`.
- **Bronze**: Ingested the source data into Delta format.
- **Silver**: Removed duplicate order IDs, validated required fields, and cleaned product names.
- **Gold**: Calculated sales amounts and generated a product-level sales summary.

**Output locations:**

- **Bronze**: `abfss://bronze@batchstorage11.dfs.core.windows.net/sales`
- **Silver**: `abfss://silver@batchstorage11.dfs.core.windows.net/sales`
- **Gold**: `abfss://gold@batchstorage11.dfs.core.windows.net/sales_summary`

**Status:** Sales data processing completed.

In [0]:
# GitHub sync test
print("My new code is ready for GitHub")